In [1]:
import sys
import os
import copy
import shutil
import cv2
import matplotlib.pyplot as plt

import numpy as np
import torch

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.predictor import ShorelinePredictor

from src.data_processing.dataset_loader import CoastData

from src.models.metrics import Metrics

In [2]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [13]:
image_type_paths = {
    "oblique": {
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_obliques_2_classes/")),
        "num_classes": 2,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment3/oblique"))
    },
    "rectified": {
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_rectified_3_classes/")),
        "num_classes": 3,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment3/rectified"))
    }
}

networks: dict[str] = {
    "DeepLabV3": {
        "weights_path": {
            "rectified_256x256": "2025-10-16-22-37-47_rectified_DeepLabV3_256x256",
            "rectified_256x512": "2025-11-03-14-58-44_rectified_DeepLabV3_256x512",
            "rectified_256x1024": "2025-11-03-17-47-12_rectified_DeepLabV3_256x1024",
            "rectified_512x512": "2025-11-03-20-29-56_rectified_DeepLabV3_512x512",
            
            "oblique_256x256": "2025-10-14-23-37-23_oblique_DeepLabV3_256x256",
            "oblique_256x512": "2025-11-11-10-04-02_oblique_DeepLabV3_256x512",
            "oblique_256x1024": "2025-11-13-08-21-39_oblique_DeepLabV3_256x1024",
            "oblique_512x512": "2025-11-14-03-14-39_oblique_DeepLabV3_512x512"
        }
    }
}

patches = {
    "256x256": {
        "patch_size": (256, 256),
        "stride": (128, 128)
    }, 
    "256x512": {
        "patch_size": (256, 512),
        "stride": (128, 256)
    },
    "256x1024": {
        "patch_size": (256, 1024),
        "stride": (128, 512)
    }, 
    "512x512": {
        "patch_size": (512, 512),
        "stride": (256, 256)
    }
}

In [15]:
for data_type in image_type_paths:
    print(f"\n{'#'*30}\nProcessing {data_type} images\n{'#'*30}")
        
    data_path = image_type_paths[data_type]["path"]
    num_classes = image_type_paths[data_type]["num_classes"]
    weights_path = image_type_paths[data_type]["weights_path"]

    print(f"Data path: {data_path}")

    # Load data
    data = CoastData(data_path)

    get_mask = True
    filtered_data = data.split_data(get_metadata=True, get_mask=get_mask)

    print(f"Number of samples: {len(filtered_data['test']['images'])}")
    for network in networks:
        for patch_name, patch_info in patches.items():
            print(f"\n{'-'*20}\nPredicting with {network} - {data_type} - {patch_name}\n{'-'*20}")
            net_weights_path = os.path.join(weights_path, networks[network]["weights_path"][f"{data_type}_{patch_name}"], "models/best_model.pth")

            predictor = ShorelinePredictor(network, net_weights_path, num_classes)

            counter = 0
            total_images = len(filtered_data['test']['images'])

            ignore_index = 0 if data_type == "rectified" else None
            average = 'weighted' #  if data_type == "rectified" else 'macro'
            metrics = Metrics(
                phase='test',
                num_classes=num_classes,
                average=average,
                compute_loss=False,
                ignore_index=ignore_index
            )

            # Predict only the test set
            for path_img, path_mask, metadata in zip(filtered_data['test']['images'], filtered_data['test']['masks'], filtered_data['test']['metadata']):
                # Get filenames
                img_filename = os.path.basename(path_img)
                mask_filename = os.path.basename(path_mask)

                gt_mask = cv2.imread(path_mask, cv2.IMREAD_GRAYSCALE)

                # Predict
                landward_pixel_pred = 1 if data_type == "rectified" else 0
                seaward_pixel_pred = 2 if data_type == "rectified" else 1
                # print(patch_info["patch_size"], patch_info["stride"])
                output = predictor.predict(path_img, patch_size=patch_info["patch_size"], stride=patch_info["stride"], landward_pixel_pred=landward_pixel_pred, seaward_pixel_pred=seaward_pixel_pred)

                pred_mask = output['predicted_mask'].astype(np.uint8)
                
                # to tensor
                gt_mask_tensor = torch.tensor(gt_mask).unsqueeze(0)
                pred_mask_tensor = torch.tensor(pred_mask).unsqueeze(0)

                metrics.update_metrics(gt_mask_tensor, pred_mask_tensor)

            metrics.compute()
            print(metrics.get_last_epoch_info())



##############################
Processing oblique images
##############################
Data path: /home/josep/LOCALDATA/Shoreline-extraction/data/processed_obliques_2_classes
CoastData: global - 1717 images
Coast: agrelo, Total size: 244
Coast: arenaldentem, Total size: 40
Coast: cadiz, Total size: 946
Coast: cies, Total size: 430
Coast: samarador, Total size: 57
Number of samples: 174

--------------------
Predicting with DeepLabV3 - oblique - 256x256
--------------------
test metrics: 
	test_accuracy: 0.9559783935546875
	test_f1_score: 0.9559805989265442
	test_precision: 0.9559870958328247
	test_recall: 0.9559783935546875
	test_confusion_matrix: 
		0.9560 0.0440
		0.0440 0.9560


--------------------
Predicting with DeepLabV3 - oblique - 256x512
--------------------
test metrics: 
	test_accuracy: 0.9603738784790039
	test_f1_score: 0.9603806734085083
	test_precision: 0.9604312181472778
	test_recall: 0.9603738784790039
	test_confusion_matrix: 
		0.9637 0.0363
		0.0426 0.9574


------